<a href="https://colab.research.google.com/github/pdemont05/BE-447-Lecture-14/blob/main/BE_447_HW_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook A: Image Classification
In this notebook, we'll apply image classification to the [STL-10 dataset](https://cs.stanford.edu/~acoates/stl10/). We'll cover data loading, preprocessing, training Multilayer Perceptron (MLP) model, and evaluating its performance. Through this exercise, you'll gain hands-on experience with fundamental machine learning workflows and image data handling.

### Set up imports

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier

### Load data
The full STL-10 dataset is very large (~2.3GB). For this notebook we will just work with the trainX.bin data. Still, the binary file for train_X.bin is too large for GitHub (136MB when the limit is 100MB). Instead download it from google drive, and put the file in the 'data' folder. https://drive.google.com/file/d/1X09rhyyAwrVgAOeOIZk0-HBqi_s5BL88/view?usp=sharing

These predefined functions will help will reading images and labels from the binary files. There is no need to modify them.

In [10]:
# Reads images from a binary file and returns them as a numpy array.
def read_images(file_path):
    image_size = 96 * 96 * 3  # Each image has 96x96 pixels with 3 channels
    with open(file_path, 'rb') as file:
        images = np.fromfile(file, dtype=np.uint8)
        images = images.reshape(-1, 3, 96, 96).transpose(0, 3, 2, 1)  # Reshape to proper format
    return images

# Reads labels from a binary file and returns them as a numpy array
def read_labels(file_path):
    with open(file_path, 'rb') as file:
        labels = np.fromfile(file, dtype=np.uint8)
    return labels

### Load images and labels using the above functions

In [ ]:
# define file path names
images_path = "train_X.bin"
labels_path = "train_y.bin"

# Load the images and labels
images = read_images(images_path)
labels = read_labels(labels_path)

In [ ]:
assert images.shape == (5000, 96, 96, 3)
assert labels.shape == (5000,)

### Print first three images

In [ ]:
# define a function to display an image
def show_image(image, label=None):
    plt.imshow(image)
    if label is not None:
        plt.title(f"Label: {label}")
    plt.axis('off')
    plt.show()

# write a loop to display the first 3 images
for i in range(3):
    show_image(images[i], label=labels[i])

###Preprocess images

In [ ]:
# inspect the image data structure
print("Images shape:", images.shape)
print("Single image shape:", images[0].shape)
print("Data type:", images.dtype)

# normalize all values to be between 0 and 1
images = images.astype('float32') / 255.0

###Perform train test split

In [ ]:
# split the data into training and test sets (80% train, 20% test) using random_state=42
labels = labels.flatten()

X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2,random_state=42)

### Define and Train the Model
First define the ML model using the `MLPClassifier` from scikit-learn. Use the hyperparameters: `hidden_layer_sizes=(512,), activation='relu', solver='adam', max_iter=10, verbose=True, random_state=42`

Note: you will need to flatten the images before training the model

In [ ]:
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

model = MLPClassifier(
    hidden_layer_sizes=(512,),
    activation='relu',
    solver='adam',
    max_iter=10,
    verbose=True,
    random_state=42
)

model.fit(X_train_flat, y_train)

### Evaluate the Model
After training, evaluate the model's performance on the test using `accuracy_score`

In [ ]:
# Predict on the test set
y_pred = model.predict(X_test_flat)

# Compute accuracy
test_accuracy = accuracy_score(y_test, y_pred)

In [ ]:
assert test_accuracy > 0.30 and test_accuracy < 0.32

### Visualize Misclassified Examples
Display the first 3 mislabeled images in `test_predictions`. Print the image, the predicted label, and the true label.

Note: the image labels are: airplane, bird, car, cat, deer, dog, horse, monkey, ship, truck

In [ ]:
# STL-10 class label names (1-indexed in the dataset)
class_names = [
    "airplane", "bird", "car", "cat", "deer",
    "dog", "horse", "monkey", "ship", "truck"
]

# Get indices of misclassified examples
misclassified_indices = np.where(y_pred != y_test)[0]

# Display first 3 misclassified images
plt.figure(figsize=(10, 4))
for i in range(3):
    idx = misclassified_indices[i]
    image = X_test[idx]
    true_label = class_names[y_test[idx] - 1]
    predicted_label = class_names[y_pred[idx] - 1]

    plt.subplot(1, 3, i + 1)
    plt.imshow(image)
    plt.title(f"True: {true_label}\nPred: {predicted_label}")
    plt.axis('off')

plt.tight_layout()
plt.show()

# Notebook B: Protein Sequence Generative AI
In this notebook we’ll apply the next‐character prediction workflow from class to biological sequences. We’ll load peroxidase sequence data from a file, preprocess the sequences, train MLPs to predict the next amino acid given a fixed context length, and then generate new peptide sequences.


### 1. Set up imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss

### 2. Load the peroxidase sequences
Only use the first 500 sequences to speed up training

In [ ]:
seqs = []

with open('peroxidase_sequences.txt', 'r') as file:
    for i, line in enumerate(file):
        if i >= 500:
            break
        seqs.append(line.strip())  # remove newline characters

In [ ]:
assert len(seqs) == 500

### 3. Build Vocabulary
Include the amino acid letters found in the sequences and the space character.

In [ ]:
all_text = " ".join(seqs)  # Add space between sequences
vocab = sorted(set(all_text))  # Unique characters

In [ ]:
assert len(vocab) == 24

### 4. Build mappings between characters and integers
Create dictionaries for converting between amino acid letters and integers

In [ ]:
char_to_int = {char: idx for idx, char in enumerate(vocab)}

int_to_char = {idx: char for char, idx in char_to_int.items()}

In [ ]:
assert int_to_char[1]

### 5. Create Training Data
For each sequence, add a number of spaces equal to the context_length at front and one space at the end. Start with a context length of 3

Create pairs of context and target letters, then convert them to integers and add to X and y data lists.

In [ ]:
context_length = 3
X = []
y = []

# Pad and encode sequences into context → target pairs
for seq in seqs:
    # Pad: 3 spaces in front, 1 space at end
    padded_seq = ' ' * context_length + seq + ' '

    # Create context-target pairs
    for i in range(len(padded_seq) - context_length):
        context = padded_seq[i:i+context_length]
        target = padded_seq[i+context_length]

        # Convert to integers
        context_int = [char_to_int[c] for c in context]
        target_int = char_to_int[target]

        X.append(context_int)
        y.append(target_int)

In [ ]:
assert len(X) == 203591


### 6. Train an MLP Classifier
Use all the default hyperparameter values. Set the random state to 42.

In [ ]:
X = np.array(X)
y = np.array(y)

clf = MLPClassifier(random_state=42)
clf.fit(X, y)

In [ ]:
assert clf.loss_ < 2.856 and clf.loss_ > 2.855

### 7. Plot the loss curve

In [ ]:
plt.plot(clf.loss_curve_)
plt.title("MLP Training Loss Curve")
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

### 8. Try training new MLPs with different hyperparameters
Try to get the log-loss value below 2.65. You might need to change the context size, hidden layer sizes, and max iterations.

In [ ]:
def generate_context_data(sequences, context_length, char_to_int):
    X, y = [], []
    for seq in sequences:
        padded = ' ' * context_length + seq + ' '  # pad front and end
        for i in range(len(padded) - context_length):
            context = padded[i:i + context_length]
            target = padded[i + context_length]
            X.append([char_to_int[c] for c in context])
            y.append(char_to_int[target])
    return np.array(X), np.array(y)

# Best-performing setup
context_length = 7
X, y = generate_context_data(seqs, context_length, char_to_int)

# Optional: limit for speed
X = X[:50000]
y = y[:50000]

clf = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    max_iter=75,
    random_state=42,
    early_stopping=True,
    verbose=True
)

clf.fit(X, y)

# Evaluate log-loss
y_probs = clf.predict_proba(X)
loss = log_loss(y, y_probs)

print(f"Log-loss: {loss:.4f}")

In [ ]:
assert clf.loss_ < 2.65

### 9. Define a function for making peptide sequences
Write a called function called generate_peptide to generate new protein sequences using the trained MLP

In [ ]:
ef generate_peptide(model, char_to_int, int_to_char, context_length, max_length=50, seed=None):
    if seed:
        seed = seed.lower()
        context = (' ' * (context_length - len(seed))) + seed
    else:
        context = ' ' * context_length

    generated = []

    for _ in range(max_length):
        # Convert context to integer vector
        context_int = [char_to_int[c] for c in context]

        # Predict next character
        next_probs = model.predict_proba([context_int])[0]
        next_idx = np.random.choice(len(next_probs), p=next_probs)
        next_char = int_to_char[next_idx]

        # Stop if space (end-of-sequence token)
        if next_char == ' ':
            break

        generated.append(next_char)

        # Update context
        context = context[1:] + next_char

    return ''.join(generated)

### 10. Generate names

Write a loop to print 10 peptide sequences

In [ ]:
for i in range(10):
    peptide = generate_peptide(
        model=clf,  # your trained MLP model
        char_to_int=char_to_int,
        int_to_char=int_to_char,
        context_length=7,  # use the context length you trained with
        max_length=50
    )
    print(f"Peptide {i+1}: {peptide}")